# 02 - Одна точка (μ, H)

Задача остановки в **одном** узле: по шагам, с промежуточными результатами.
Здесь видно, что именно делает `value_at` внутри.

Генерация fBm разобрана в `01-fbm.ipynb`, перебор сетки - в `03-grid.ipynb`.

In [ ]:
import numpy as np
from osfbm.config import Config
from osfbm import adapter
from osfbm.core import value_at

import matplotlib.pyplot as plt


C1, C2, C3 = "#2a78d6", "#eb6834", "#1baf7a"
INK, MUTED = "#0b0b0b", "#8a8a86"

plt.rcParams.update({
    "figure.figsize": (7.5, 4.2), "figure.dpi": 110,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": MUTED, "axes.labelcolor": INK, "axes.titlesize": 11,
    "axes.grid": True, "grid.color": "#e8e8e4", "grid.linewidth": 0.8,
    "lines.linewidth": 1.8, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "font.size": 9,
    "legend.frameon": False,
})

## Параметры этого расчёта

Полная таблица полей `Config` - в `01-fbm.ipynb`. Здесь важны эти:

| Поле | Роль в задаче остановки |
|---|---|
| `n_exercise` | сколько раз разрешено продать. Обучается `n_exercise - 1` регрессий: на последней дате остановка принудительна, решать нечего |
| `n_fine` | густота сетки для интегралов сигнатуры. На точность признаков влияет, на число решений - нет |
| `K` | уровень усечения сигнатуры, признаков $2^{K+1}-1$. Главный компромисс «выразительность против переобучения» |
| `M_train` / `M_test` | обучение и оценка **на разных** выборках - условие, без которого оценка снизу перестаёт быть оценкой снизу |
| `ridge` | регуляризация регрессии continuation value |

Аргументы самого расчёта - два числа:

| Аргумент | Значение |
|---|---|
| `mu` | нормированный снос $\mu = a/\sigma$. Единственный параметр сноса в задаче: исходные $a$ и $\sigma$ входят только через отношение |
| `H` | параметр Херста, задаёт автокорреляцию приращений |

In [ ]:
cfg = Config(T=1.0, n_exercise=20, n_fine=100, M_train=10_000, M_test=10_000, K=4)
MU, H = 0.5, 0.3

print(f"конфиг {cfg.hash}")
print(f"дат решения: {cfg.n_exercise}, мелкая сетка: {cfg.n_fine}, K = {cfg.K}")
print(f"первые даты решения: {cfg.exercise_times[:4]} ... последняя {cfg.exercise_times[-1]}")

## Шаг 1 - траектории

$B^H$ от $\mu$ **не зависит**: снос входит только в $X = \mu t + B^H$.
Поэтому траектории генерируются один раз на $H$ и переиспользуются для всех
$\mu$ - это common random numbers из `docs/04.3`, здесь они достаются даром.

In [ ]:
paths = adapter.simulate_pair(H, cfg)
print({k: v.shape for k, v in paths.items()})

## Шаг 2 0 добавляем снос

Единственная содержательная правка постановки относительно статьи
(`docs/04.1`): $B^H_t \longrightarrow X_t = \mu t + B^H_t$.
Дальше $X$ идёт и в сигнатуры, и в payoff.

`adapter.drifted(B, mu, cfg)` - принимает массив траекторий, снос и конфиг;
возвращает массив той же формы.

In [ ]:
X_tr = adapter.drifted(paths["B_train"], MU, cfg)
X_te = adapter.drifted(paths["B_test"], MU, cfg)

fig, ax = plt.subplots()
for p in X_te[:40]:
    ax.plot(cfg.times, p, color=C1, alpha=0.25, linewidth=1.0)
ax.plot(cfg.times, MU * cfg.times, color=C2, linewidth=2.2, label=f"снос  μt,  μ={MU}")
ax.axhline(0, color=MUTED, linewidth=0.8)
ax.set_xlabel("t"); ax.set_ylabel("$X_t$")
ax.set_title(f"40 траекторий $X_t = μt + B^H_t$   (H = {H})", loc="left")
ax.legend()
plt.tight_layout()

## Шаг 3 - сигнатуры

Сигнатура кодирует **историю** траектории конечномерным вектором: fBm при
$H\ne1/2$ немарковский, поэтому continuation value зависит от всего пути, а не
от текущего значения.

`adapter.signatures(X, cfg)` считает сигнатуру расширенного пути $(t, X_t)$ во
всех точках мелкой сетки и возвращает куб
`(траектории, моменты, признаки)`. Признаков $2^{K+1}-1$.

In [ ]:
sig_tr = adapter.signatures(X_tr, cfg)
sig_te = adapter.signatures(X_te, cfg)

print(f"куб сигнатур: {sig_tr.shape}  = (траектории, моменты, признаки)")
print(f"признаков: {sig_tr.shape[2]}, ожидаем 2^(K+1)-1 = {cfg.sig_dim}")
print()
print("нулевой уровень — константа 1:", np.allclose(sig_tr[:, :, 0], 1.0))
print("уровень 1, компонента t  :", sig_tr[0, -1, 1], " (должно быть T =", cfg.T, ")")
print("уровень 1, компонента X  :", sig_tr[0, -1, 2], " (должно быть X_T =", X_tr[0, -1], ")")

## Шаг 4 - Longstaff–Schwartz

Обратная рекурсия: на каждой дате решения continuation value приближается
линейной регрессией по сигнатуре, дальше сравнивается с текущим reward.

`adapter.longstaff_schwartz(sig_tr, X_tr, sig_te, X_te, cfg)` возвращает
`(V_raw, СКО, regr)`. Обучение идёт на первой паре аргументов, оценка — на
второй, **независимой**. Это несущее условие: иначе момент остановки перестаёт
быть stopping time и оценка снизу ломается (`docs/03.4`).

In [ ]:
v_raw, sample_std, regr = adapter.longstaff_schwartz(sig_tr, X_tr, sig_te, X_te, cfg)
print(f"V_raw = {v_raw:.6f}")
print(f"обучено регрессий: {len(regr)}  (= число дат решения минус 1)")
print(f"коэффициентов в каждой: {regr[0].coef_.shape[0]}")

## Шаг 5 — моменты остановки

`price` авторов возвращает только значение, самих $\tau$ там нет. Мы
восстанавливаем их из обученных регрессий тем же правилом, но векторно:
`adapter.stopping_indices(regr, sig_te, X_te, cfg)` даёт индексы дат остановки
в диапазоне `0..n_exercise-1`, а сами моменты — это `cfg.exercise_times[idx]`.

Проверка: средний reward в восстановленных $\tau$ обязан совпасть со значением
авторов **в точности**.

In [ ]:
idx = adapter.stopping_indices(regr, sig_te, X_te, cfg)
tau = cfg.exercise_times[idx]

v_ours = X_te[np.arange(len(idx)), cfg.exercise_index[idx]].mean()
print(f"V авторов      = {v_raw:.10f}")
print(f"V из наших tau = {v_ours:.10f}")
print("совпадает бит-в-бит:", np.isclose(v_raw, v_ours, atol=1e-12))

fig, ax = plt.subplots(figsize=(7.5, 3.4))
ax.hist(tau, bins=cfg.n_exercise, color=C1, edgecolor="white", linewidth=1.2)
ax.set_xlabel("$τ^*$"); ax.set_ylabel("траекторий")
ax.set_title(f"Распределение момента остановки   (μ={MU}, H={H})", loc="left")
plt.tight_layout()
print(f"\nE[tau] = {tau.mean():.4f},  p1 = {np.mean(idx == cfg.n_exercise - 1):.4f}")

## Всё сразу: `value_at`

`value_at(mu, H, cfg, paths=None)` — единственная точка входа для одного узла;
внутри те же пять шагов плюс поправка на нулевой момент. Аргумент `paths`
необязательный: если передать готовый словарь из `simulate_pair`, траектории
не будут генерироваться заново. Именно так устроен скан по $\mu$.

Поля результата:

| Поле | Значение |
|---|---|
| `V` | оценка $\hat V(\mu,H)$ **с** поправкой на нулевой момент |
| `V_raw` | она же до поправки, то есть значение задачи на $t_1..t_N$. Отрицательное `V_raw` запускает текущую поправку выхода в нуле; это не доказательство оптимальности |
| `SE` | стандартная ошибка Монте-Карло, $\text{СКО}/\sqrt{M_{\text{test}}}$ |
| `p0` | доля траекторий с $\tau^*=0$. Принимает только значения 0 и 1 |
| `p1` | доля траекторий, удержанных до конца горизонта |
| `E_tau` | средний момент остановки; по теореме об огибающей равен наклону $\partial_\mu V$ |
| `stopped_at_zero` | флаг режима «продать сразу» |
| `config_hash` | отпечаток конфига, которым получен результат |

In [ ]:
r = value_at(MU, H, cfg, paths=paths)
for k, v in r.as_row().items():
    print(f"  {k:16s} {v}")

## Поправка на нулевой момент

Даты остановки в коде авторов начинаются с $t_1$ — продать в нулевой момент их
код не даёт. А наш левый режим — это ровно $\tau^*=0$.

Так как $\mathcal F_0$ тривиальна, continuation value в нуле равен значению
задачи на $t_1..t_N$, а $Z_0 = 0$ известен заранее. Для истинного ожидаемого продолжения верно $V=\max(0,V_{\mathrm{continuation}})$. В коде
`V_raw` — тестовое среднее; его обрезка снизу нулём может вносить смещение.
Исправление выбора стратегии в нуле остаётся отдельной задачей.

Следствие: решение в нуле одинаково для всех траекторий, поэтому $\hat p_0$
принимает только значения 0 и 1.

In [ ]:
for mu in (-3.0, -1.5, -0.5, 0.5):
    rr = value_at(mu, H, cfg, paths=paths)
    flag = "STOP в t_0" if rr.stopped_at_zero else ""
    print(f"μ={mu:+5.1f}   V_raw={rr.V_raw:+.5f}   ->  V={rr.V:.5f}   p0={rr.p0:.0f}  {flag}")

## Сверка с Section 4.1 статьи

Опорная точка: $\mu=0$ воспроизводит задачу авторов $\sup_\tau E[B^H_\tau]$.
Их notebook при $H=0.3$, $N=100$, $N_1=10$, $K=4$, $M=50000$ даёт
**0.2436758 ± 0.0035850**.

In [ ]:
from osfbm.checks import REFERENCE_MU0

ref_cfg = Config(n_exercise=10, n_fine=100, M_train=50_000, M_test=50_000, K=4)
r_ref = value_at(0.0, REFERENCE_MU0["H"], ref_cfg)

comb = np.hypot(r_ref.SE, REFERENCE_MU0["SE"])
print(f"наш    {r_ref.V:.7f} ± {r_ref.SE:.7f}")
print(f"эталон {REFERENCE_MU0['V']:.7f} ± {REFERENCE_MU0['SE']:.7f}")
print(f"расхождение {abs(r_ref.V - REFERENCE_MU0['V']) / comb:.2f} комбинированных SE")

## Сохранение результата

Узел пишется в CSV через `osfbm.results.append(path, rows, key)`:

| Аргумент | Значение |
|---|---|
| `path` | файл CSV; недостающие директории создаются |
| `rows` | список словарей — ровно то, что отдаёт `NodeResult.as_row()` |
| `key` | по каким колонкам строки считаются одной и той же точкой |

Дозапись **идемпотентна по ключу**: повторный прогон той же точки перезаписывает
строку, а не плодит дубликат. Для узлов ключ — `config_hash, H, mu`. Отпечаток
конфига в ключе нужен, чтобы прогоны с разными `K`, `M`, `n_fine` не затирали
друг друга: та же пара $(\mu, H)$ при других настройках ложится отдельной строкой.

Результаты сохраняются в `result_optimal_stopping/` в корне проекта.

Для сетки используйте `03-grid.ipynb`, для анализа сохранённого запуска — `04-grid-analysis.ipynb`.

In [ ]:
from pathlib import Path
import pandas as pd
from osfbm import results

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
NODES = ROOT / "result_optimal_stopping" / "nodes.csv"

results.append(NODES, [r.as_row(), r_ref.as_row()], key=["config_hash", "H", "mu"])
results.append(NODES, [r.as_row()], key=["config_hash", "H", "mu"])

saved = pd.read_csv(NODES)
print(f"{NODES}")
print(f"  всего {len(saved)} строк, конфигов: {saved['config_hash'].nunique()}")

mine = saved[saved["config_hash"].isin({r.config_hash, r_ref.config_hash})]
mine[["config_hash", "H", "mu", "V", "SE", "p0", "p1", "E_tau"]].round(5)